# 03 · Preprocessing (Member A)

Cleans the LBNL chiller plant data and adds the team's derived metrics.
**Every file gets exactly the same steps** (settings in `config.yaml` → `preprocess:`):

1. Frozen-sensor removal: off (same reason as 3-sigma)
2. No 3-sigma removal (`outlier_removal: false`): the data are simulated, so there are no sensor errors, and on fault files the test would delete the fault itself
3. 15-minute means
4. Keep only windows where at least one chiller is on
5. Derived metrics: `CHL_DT_i`, `CHL_COOL_i`, `CHL_COP_i`, `CHL_CDDT_i`, `CT_APPROACH_i`, `PLANT_POW`, `PLANT_COP`

**What this notebook shows:** rows before and after cleaning · hours with chillers on per month · histograms of `CHL_COP_1` and `PLANT_COP` · an energy-balance check of the cooling formula.

Set `DATA_DIR` in your `.env` to the folder with the 24 CSV files.

In [ ]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":          # run from the repo root
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import matplotlib.pyplot as plt
from src import preprocess as pp

S = pp.load_settings("config.yaml")
pd.set_option("display.width", 140)
{k: S[k] for k in ["frozen_removal", "frozen_window", "outlier_removal", "resample",
                   "min_coverage", "chiller_on_only", "min_cool_kw", "delta_t_scale"]}

## Pick the files
The fault-free year and one fault file, processed with the same steps.

In [ ]:
files = pp.list_files(pp.data_dir(S["raw_dir"]))
FAULT_FREE = pp.find_fault_free(files, S["fault_free_file"])      # ChillerPlant.csv
FAULT_FILE = next((f for f in files if "coolingtower_fouling" in f.name),
                  next(f for f in files if f != FAULT_FREE))      # change to any scenario
LIMITS = pp.fit_limits(pp.load_csv(FAULT_FREE), S) if S["outlier_removal"] else None
print(len(files), "CSV files found")
print("fault-free :", FAULT_FREE.name)
print("example    :", FAULT_FILE.name)
print("3-sigma step:", "ON (all files)" if S["outlier_removal"] else "OFF (all files)")

## 1 · Rows before and after cleaning

In [ ]:
results = {}
for f in [FAULT_FREE, FAULT_FILE]:
    raw, full, rep, saved = pp.process_file(f, LIMITS, S)
    results[f.name] = (raw, full, rep)
    print("saved", saved)

summary = pd.DataFrame({
    name: {
        "raw 1-min rows": r["rows_raw"],
        "after frozen-sensor removal": r["rows_after_frozen"],
        "after 3-sigma removal": r["rows_after_outliers"],
        "15-min windows": r["rows_15min"],
        "windows with a chiller on": r["rows_chiller_on"],
        "% of 1-min rows kept before resampling": round(100 * r["rows_after_outliers"] / r["rows_raw"], 2),
    } for name, (_, _, r) in results.items()
}).T
summary

In [ ]:
# Which sensors caused the frozen-sensor removals (same rule for both files)
for name, (_, _, r) in results.items():
    print(f"=== {name}")
    fz = r.get("frozen_by_column", pd.Series(dtype=int))
    print("frozen, top columns:\n", fz.head(8).to_string() if len(fz) else "  none")
    if r.get("frozen_check_skipped"):
        print("frozen check skipped (constant by design, e.g. fixed-speed pumps):",
              sorted(r["frozen_check_skipped"]))
    print()

## 2 · Hours with chillers on, per month
From the raw 1-minute fault-free year.

In [ ]:
ff_raw = results[FAULT_FREE.name][0]
sta = ff_raw[[f"CHL_STA_{i}" for i in range(1, S["n_chillers"] + 1)]] > 0.5
hours = sta.groupby(sta.index.month).sum() / 60          # 1-minute data -> hours
hours.index = [pd.Timestamp(2018, m, 1).strftime("%b") for m in hours.index]
hours.columns = [f"Chiller {i}" for i in range(1, S["n_chillers"] + 1)]

ax = hours.plot.bar(figsize=(11, 4.5), width=0.8)
ax.set_ylabel("hours on"); ax.set_xlabel("")
ax.set_title("Hours with each chiller on, per month (fault-free year)")
ax.grid(axis="y", alpha=.3); plt.xticks(rotation=0); plt.tight_layout(); plt.show()
print("Total hours on:", hours.sum().round(1).to_dict())
hours.round(1).T

## Energy-balance check of the cooling formula
`CHL_COOL_i = flow (GPM) × ΔT (°F) × 500 / 3412.14` (kW). The cooling made by the chillers should roughly equal the building load `CWL_SEC_LOAD / 1000` (kW), so the ratio below should be close to 1. A ratio near 0.56 would mean the temperatures are in °C (set `delta_t_scale: 1.8`).

In [ ]:
ff = results[FAULT_FREE.name][1]
ratio = pp.energy_balance(ff, S)
print(ratio.round(3).to_string())
cool = sum(ff[f"CHL_COOL_{i}"] for i in range(1, S["n_chillers"] + 1))
load = ff["CWL_SEC_LOAD"] / 1000
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(load, cool, s=3, alpha=.3)
m = float(max(load.max(), cool.max()))
ax.plot([0, m], [0, m], "k--", lw=1, label="equal")
ax.set_xlabel("building load CWL_SEC_LOAD/1000 (kW)"); ax.set_ylabel("sum of CHL_COOL_i (kW)")
ax.set_title("Chiller cooling vs building load (15-min)"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 3 · COP histograms
Only windows with real cooling (≥ `min_cool_kw`). A healthy chiller is roughly 4–6; plant COP is lower because pumps and fans are included.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, col, colr in zip(axes, ["CHL_COP_1", "PLANT_COP"], ["#2f6fd6", "#e8902a"]):
    x = ff[col].dropna()
    ax.hist(x.clip(0, 10), bins=50, range=(0, 10), color=colr, alpha=.85)
    ax.axvline(x.median(), color="k", ls="--", lw=1)
    ax.set_title(f"{col}  (median {x.median():.2f}, n={len(x):,} windows)")
    ax.set_xlabel("COP"); ax.set_ylabel("15-min windows"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Derived columns
Names agreed with the team (Member B uses them as graph nodes).

In [ ]:
derived = [c for c in ff.columns
           if c.startswith(("CHL_DT_", "CHL_COOL_", "CHL_COP_", "CHL_CDDT_", "CT_APPROACH_"))
           or c in ("PLANT_POW", "PLANT_COP")]
print("negative CT_APPROACH values:",
      {c: int((ff[c] < 0).sum()) for c in derived if c.startswith("CT_APPROACH_")})
ff[derived].describe().T.round(2)

## Process all 24 files
Same steps for every file; output in `data/processed/`. Takes a while. Same as running `python -m src.preprocess`.

In [5]:
RUN_ALL = False
if RUN_ALL:
    table = pp.process_all(S)
    table

ChillerPlant_secondary_chilled_water_pressure_bias_-020.csv: 525540 -> 32993 rows, saved data\processed\ChillerPlant_secondary_chilled_water_pressure_bias_-020.parquet


KeyboardInterrupt: 

In [ ]:
%pip install --upgrade --no-cache-dir pyarrow

In [ ]:
# Setup (works even if the cells above were not run)
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))
import numpy as np
import pandas as pd
from src import preprocess as pp

S = pp.load_settings("config.yaml")
files = pp.list_files(pp.data_dir(S["raw_dir"]))
FAULT_FREE = pp.find_fault_free(files, S["fault_free_file"])

parquet = Path(S["processed_dir"]) / "ChillerPlant.parquet"
if parquet.exists():
    raw = pp.load_csv(FAULT_FREE)
    proc = pd.read_parquet(parquet)
else:
    raw, proc, _, _ = pp.process_file(FAULT_FREE, None, S)

# 1. Wet-bulb can never be above dry-bulb
wb_above = (raw["OA_TEMP_WB"] > raw["OA_TEMP"] + 0.01).mean()
print(f"1. OA_TEMP_WB > OA_TEMP in {100*wb_above:.2f}% of minutes")
print(raw[["OA_TEMP", "OA_TEMP_WB"]].describe().round(1).T, "\n")

# 2. Does the tower filter do anything? (flow and fan power at 0?)
for c in ["CT_STA_1", "CT_FLOW_1", "CT_POW_1", "CT_FAN_SPD_1"]:
    x = raw[c]
    print(f"2. {c:<13} min {x.min():8.2f}  max {x.max():8.2f}  "
          f"share == 0: {100*(x == 0).mean():5.1f}%   unique values: {x.nunique()}")
print()

# 3. Leaving water vs its set-point, in windows where approach < 0
neg = proc["CT_APPROACH_1"] < 0
cols = ["CT_SW_TEMP_1", "CT_SW_TEMPSPT", "OA_TEMP_WB", "OA_TEMP",
        "CT_FLOW_1", "CT_POW_1", "CT_FAN_SPD_1", "TWV_CTRL"]
print(f"3. {neg.sum()} windows with negative approach. Typical values there vs elsewhere:")
print(pd.DataFrame({"approach < 0": proc.loc[neg, cols].median(),
                    "approach >= 0": proc.loc[~neg & proc["CT_APPROACH_1"].notna(), cols].median()}).round(2))
print("\nMonths with negative approach:", proc.index[neg].month.value_counts().sort_index().to_dict())

In [ ]:
dry = raw["OA_TEMP_WB"]          # what we think is the real dry-bulb
spt = raw["CWL_PRI_SW_TEMPSPT"]
print("set-point when 'OA_TEMP_WB' <= 60:", spt[dry <= 60].median().round(1), "(expect ~52)")
print("set-point when 'OA_TEMP_WB' >= 80:", spt[dry >= 80].median().round(1), "(expect ~42)")
print("same test with OA_TEMP:", spt[raw['OA_TEMP'] <= 60].median().round(1),
      spt[raw['OA_TEMP'] >= 80].median().round(1))

In [ ]:
def reset_error(x):
    pred = (54 - (x - 60) * 0.5).clip(44, 54)        # 54 F at 60 F outdoor -> 44 F at 80 F
    return (pred - spt).abs().mean().round(2)
print("mean error using OA_TEMP_WB as dry-bulb:", reset_error(raw["OA_TEMP_WB"]))
print("mean error using OA_TEMP    as dry-bulb:", reset_error(raw["OA_TEMP"]))

In [ ]:
# Setup (works even if the cells above were not run)
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))
import numpy as np
import pandas as pd
from src import preprocess as pp

S = pp.load_settings("config.yaml")
files = pp.list_files(pp.data_dir(S["raw_dir"]))
FAULT_FREE = pp.find_fault_free(files, S["fault_free_file"])

parquet = Path(S["processed_dir"]) / "ChillerPlant.parquet"
if parquet.exists():
    raw = pp.load_csv(FAULT_FREE)
    proc = pd.read_parquet(parquet)
else:
    raw, proc, _, _ = pp.process_file(FAULT_FREE, None, S)

# 1. Wet-bulb can never be above dry-bulb
wb_above = (raw["OA_TEMP_WB"] > raw["OA_TEMP"] + 0.01).mean()
print(f"1. OA_TEMP_WB > OA_TEMP in {100*wb_above:.2f}% of minutes")
print(raw[["OA_TEMP", "OA_TEMP_WB"]].describe().round(1).T, "\n")

# 2. Does the tower filter do anything? (flow and fan power at 0?)
for c in ["CT_STA_1", "CT_FLOW_1", "CT_POW_1", "CT_FAN_SPD_1"]:
    x = raw[c]
    print(f"2. {c:<13} min {x.min():8.2f}  max {x.max():8.2f}  "
          f"share == 0: {100*(x == 0).mean():5.1f}%   unique values: {x.nunique()}")
print()

# 3. Leaving water vs its set-point, in windows where approach < 0
neg = proc["CT_APPROACH_1"] < 0
cols = ["CT_SW_TEMP_1", "CT_SW_TEMPSPT", "OA_TEMP_WB", "OA_TEMP",
        "CT_FLOW_1", "CT_POW_1", "CT_FAN_SPD_1", "TWV_CTRL"]
print(f"3. {neg.sum()} windows with negative approach. Typical values there vs elsewhere:")
print(pd.DataFrame({"approach < 0": proc.loc[neg, cols].median(),
                    "approach >= 0": proc.loc[~neg & proc["CT_APPROACH_1"].notna(), cols].median()}).round(2))
print("\nMonths with negative approach:", proc.index[neg].month.value_counts().sort_index().to_dict())

In [ ]:
for name in ["ChillerPlant_bypass_stuck_075.csv", "ChillerPlant_coolingtower_bias_2.csv"]:
    f = next(x for x in files if x.name == name)
    _, _, rep, _ = pp.process_file(f, None, S)
    print(f"=== {name}: {rep['rows_raw']} -> {rep['rows_after_frozen']} minutes after frozen removal")
    print(rep["frozen_by_column"].head(8).to_string(), "\n")